# ECON2041 week 10 lecture: can we trust these data?

The pokies data are the 79 Victorian local government areas (LGAs) from weeks 6 to 9.
Today we'll also use a copy of the same data with some problems that we planted.

## Data sources

- Gaming expenditure: Victorian Gambling and Casino Control Commission, 2024
- Employment: Department of Employment, Small Area Labour Markets, 2024
- Population: Department of Environment, Land, Water and Planning, 2023
- Disadvantage: Australian Bureau of Statistics, Socio-Economic Indexes for Areas (SEIFA)

## Setup

Our usual setup block, the same as in week 9.

In [ ]:
# Our standard ECON2041 setup block
import numpy as np               # numerical tools (nicknamed np)
import pandas as pd              # data tools (nicknamed pd)
import matplotlib.pyplot as plt  # plotting tools (nicknamed plt)
import seaborn as sns            # statistical charts (nicknamed sns)
from statsmodels.formula.api import ols  # ordinary least squares regression
from statsmodels.iolib.summary2 import summary_col  # several models in one table

# Keep scalar output plain under NumPy 2 (0.5, not np.float64(0.5)).
if np.lib.NumpyVersion(np.__version__) >= "2.0.0":
    np.set_printoptions(legacy="1.25")

# Plot styling, set once here so every figure below inherits it
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["figure.figsize"] = (10, 6)

DATA = "https://emiliatjernstrom.com/econ2041/data"   # Unit datasets live at this web address

print("Setup done!")

## Load the data

We'll use these variables:

| Variable | What it stores |
|---|---|
| `exp_per_adult` | Gambling expenditure (losses) per adult (\$/year) |
| `egm_per_1000` | Electronic gaming machines (EGMs, or pokies) per 1,000 adults, which we call pokie density |
| `ue_rate` | LGA unemployment rate, as a proportion |
| `SEIFA_dis_score` | SEIFA disadvantage score (higher = less disadvantaged) |

`pokies` stores the clean data, and `messy` stores the copy with the planted problems.

In [ ]:
pokies = pd.read_csv(f"{DATA}/pokies-victoria-2024.csv")
messy = pd.read_csv(f"{DATA}/pokies-victoria-2024-messy.csv")

print("pokies:", pokies.shape)
print("messy:", messy.shape)

## We planted problems in the pokies data

`describe()` gives the count, mean, standard deviation, min, max, and quartiles of each variable.
Look for anything that can't be right.

In [ ]:
messy[["exp_per_adult", "egm_per_1000", "ue_rate",
       "SEIFA_dis_score"]].describe().round(2)

## Histograms show values far from the rest

`describe()` gives 1 min and 1 max per variable, and a histogram shows every value at once.

In [ ]:
sns.histplot(data=messy, x="exp_per_adult", bins=11, edgecolor="black")

In [ ]:
sns.histplot(data=messy, x="ue_rate", bins=11, edgecolor="black")

## One line of Python finds each planted problem

`.loc[condition, "LGA"]` keeps the LGA names of the rows where the condition is true, and `.tolist()` prints them as a list.

Losses per adult can't be negative:

In [ ]:
messy.loc[messy["exp_per_adult"] < 0, "LGA"].tolist()

`duplicated()` is `True` for each row whose LGA already appeared higher up in the file:

In [ ]:
messy.loc[messy["LGA"].duplicated(), "LGA"].tolist()

An unemployment rate stored as a proportion can't be above 1:

In [ ]:
messy.loc[messy["ue_rate"] > 1, "LGA"].tolist()

## Do the extra variables improve the fit?

The unrestricted model regresses losses per adult on pokie density, the high-unemployment dummy, and the SEIFA disadvantage score.
The restricted model keeps only pokie density.
The null hypothesis is $H_0: eta_2 = eta_3 = 0$, that the coefficients on the dummy and on the disadvantage score are both 0.
If the null is true, dropping them should barely hurt the fit.

We build the high-unemployment dummy the same way as in week 8.

In [ ]:
median_val = pokies["ue_rate"].median()
pokies["high_ue"] = (pokies["ue_rate"] > median_val).astype(int)
model = ols("exp_per_adult ~ egm_per_1000 + high_ue + SEIFA_dis_score",
            data=pokies).fit()
restricted = ols("exp_per_adult ~ egm_per_1000", data=pokies).fit()

## Dropping variables leaves more unexplained

The sum of squared residuals (SSR) is what each model leaves unexplained.
`.ssr` stores a model's SSR.

In [ ]:
print(f"Restricted SSR: {restricted.ssr:,.0f}")
print(f"Unrestricted SSR: {model.ssr:,.0f}")
print(f"Difference: {restricted.ssr - model.ssr:,.0f}")

## F compares the loss of fit with the noise

$$F = rac{(SSR_r - SSR_{ur})/Q}{SSR_{ur}/(n-k-1)}$$

$Q = 2$ is the number of restrictions, and `.df_resid` stores $n-k-1$.

In [ ]:
Q = 2
loss = (restricted.ssr - model.ssr) / Q
noise = model.ssr / model.df_resid

print(f"Loss of fit per restriction: {loss:,.0f}")
print(f"Noise per degree of freedom: {noise:,.0f}")
print("F:", round(loss / noise, 2))

## We reject the null that both coefficients are 0

`f_test` takes both restrictions inside one pair of quotation marks.

In [ ]:
model.f_test("high_ue = 0, SEIFA_dis_score = 0")

## Duplicates make us look more sure than we are

`pd.concat` stacks the pokies data on top of itself, so every LGA appears twice.

In [ ]:
twice = pd.concat([pokies, pokies])
model_twice = ols("exp_per_adult ~ egm_per_1000 + high_ue + SEIFA_dis_score",
                  data=twice).fit()
model_twice.f_test("high_ue = 0, SEIFA_dis_score = 0")

Same coefficients, smaller standard errors:

In [ ]:
print("high_ue coefficient:", round(model_twice.params["high_ue"], 2),
      "against", round(model.params["high_ue"], 2))
print("high_ue standard error:", round(model_twice.bse["high_ue"], 2),
      "against", round(model.bse["high_ue"], 2))

## With the -999 left in, we can't reject the null

First we fix 2 of the 3 problems in the messy file.
`drop_duplicates()` drops each row that is an exact copy of a row above it, and we divide the unemployment rate above 1 by 100.
The `-999` stays in for now.

In [ ]:
messy = messy.drop_duplicates()
messy.loc[messy["ue_rate"] > 1, "ue_rate"] = messy["ue_rate"] / 100

median_val = messy["ue_rate"].median()
messy["high_ue"] = (messy["ue_rate"] > median_val).astype(int)
model_999 = ols("exp_per_adult ~ egm_per_1000 + high_ue + SEIFA_dis_score",
                data=messy).fit()
model_999.f_test("high_ue = 0, SEIFA_dis_score = 0")

In [ ]:
print("high_ue coefficient:", round(model_999.params["high_ue"], 2),
      "against", round(model.params["high_ue"], 2))

## With -999 as missing, we reject the null again

`replace` turns every `-999` into `NaN`, Python's code for a missing value.
Never replace a missing value with 0: Whittlesea's losses aren't 0, we just don't know them.
`ols` drops any row with a missing value.

In [ ]:
messy = messy.replace(-999, np.nan)

median_val = messy["ue_rate"].median()
messy["high_ue"] = (messy["ue_rate"] > median_val).astype(int)
model_cleaned = ols("exp_per_adult ~ egm_per_1000 + high_ue + SEIFA_dis_score",
                    data=messy).fit()
print("n =", int(model_cleaned.nobs))
model_cleaned.f_test("high_ue = 0, SEIFA_dis_score = 0")

## F is larger with duplicates, smaller with -999

`.fvalue` stores the F statistic from `f_test`.

In [ ]:
hypothesis = "high_ue = 0, SEIFA_dis_score = 0"
pd.Series({
    "Clean": model.f_test(hypothesis).fvalue,
    "Every LGA twice": model_twice.f_test(hypothesis).fvalue,
    "One -999": model_999.f_test(hypothesis).fvalue,
    "Cleaned": model_cleaned.f_test(hypothesis).fvalue,
}).round(2)

## Your turn

If you want to try some other things:

1. Run `describe()` on the clean `pokies` data and compare the count, min, and max with the messy file
2. Run `pokies["LGA"].duplicated().sum()` and check that the clean data have no repeated LGAs
3. Before you run any test in your problem set, run `describe()`, `histplot()`, and `duplicated()` on your dataset